# 01 — Loading the Knowledge Library

`load()` parses every entry under `entries/`, validates each one against the
Pydantic models in `src/coralmap/models/`, and returns a frozen
`KnowledgeLibrary`.

This notebook loads the checkout on disk rather than the copy bundled in the
installed package, so you can edit an entry's YAML and reload without
reinstalling. `load()` is cached per path, so call `coralmap.clear_cache()`
before reloading; the last section shows how.

## Find the checkout

`load(path)` reads the path you pass, then `$CORALMIND_KL_PATH`, then the
entries bundled in the installed package.

In [ ]:
from pathlib import Path


def repo_root() -> Path:
    """The checkout holding ``entries/``, whether Jupyter started in ``notebooks/`` or the root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "entries").is_dir() and (candidate / "src" / "coralmap").is_dir():
            return candidate
    raise FileNotFoundError("could not find the knowledge-library checkout")


REPO = repo_root()
REPO.name

## A strict load

`load()` has no lenient mode. If any entry fails validation it raises
`LibraryParseError`, whose `problems` lists every failure. The shipped
entries all parse.

In [ ]:
from coralmap import LibraryParseError, load

try:
    kl = load(REPO)
except LibraryParseError as error:
    print(f"{len(error.problems)} entries did not validate")
    print(error.problems[0].splitlines()[0])
else:
    print(f"{len(kl.all_entries())} entries and {len(kl.sources)} sources loaded")

## Version and digest

In [ ]:
print("version     ", kl.version)  # package release
print("digest      ", kl.digest)  # content hash

`kl.version` is the package release. `kl.digest` is a hash of the entries
and sources themselves, so the same content gives the same digest wherever it
lives. A working copy loaded from disk still reports the installed package's
version, so cite the digest.

## What loaded


In [ ]:
print(f"{len(kl.all_entries()):3} entries in total")
print(f"{len(kl.pathways):3} pathways")
print(f"{len(kl.behaviors):3} AI behaviors")
print(f"{len(kl.user_behaviors):3} user behaviors")
print(f"{len(kl.impacts):3} impacts")

## Every failure at once: `parse_problems`

`parse_problems()` returns one message per file that fails validation,
without raising. An empty list means `load()` will succeed.

In [ ]:
from coralmap import parse_problems

problems = parse_problems(REPO)
print(f"{len(problems)} files failed")
for problem in problems:
    print(problem.splitlines()[0])

## Reloading after an edit

These notebooks pass `REPO` explicitly. In a long-running session you can
instead set

```bash
export CORALMIND_KL_PATH=/path/to/knowledge-library
```

and call `load()` with no argument.

`load()` is cached per resolved path, so calling it again after you edit a
YAML file returns the library you already had. Call `clear_cache()` first to
re-read `entries/` from disk.

In [ ]:
from coralmap import clear_cache

print("cached:", load(REPO) is kl)

clear_cache()
reloaded = load(REPO)
print("re-read:", reloaded is not kl)
print("digest:", reloaded.digest)

Next: [02_querying_and_traversal.ipynb](02_querying_and_traversal.ipynb) —
filters, `leading_to()`, and resolved pathway views.
